# Projet BGES d’une Organisation

In [20]:
import os
import subprocess

os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
# Spark 3.5.x requires Java 17 (Java 24+ breaks Hadoop security APIs)
os.environ["JAVA_HOME"] = subprocess.check_output(
    ["/usr/libexec/java_home", "-v", "17"], text=True
).strip()

# masque les warnings lors de la creation de l'etl
import warnings
from pyspark.pandas.utils import PandasAPIOnSparkAdviceWarning
warnings.filterwarnings("ignore", category=PandasAPIOnSparkAdviceWarning)

In [21]:
import pandas as pd
import numpy as np
import unicodedata # pour la normalisation des accents
from datetime import datetime, date, timedelta
import asyncio

import pyspark.pandas as ps
from pyspark.sql import SparkSession
from pyspark.sql import Row
from numpy._core.multiarray import empty_like
from sklearn.linear_model import LinearRegression

In [22]:
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")
from pyspark.sql.functions import *

In [23]:
# ==============================================================================
# CONFIGURATION (codé en dur)
# ==============================================================================

BASE_DIR = os.path.join(os.path.dirname("BDD_BGES"), "BDD_BGES/BDD_BGES")

SITES = ["BERLIN", "LONDON", "LOSANGELES", "NEWYORK", "PARIS", "SHANGHAI"]

IMPACT_PATH = os.path.join(BASE_DIR, "materiel_informatique_impact.csv")
DISTANCE_REF_PATH = os.path.join(BASE_DIR, "referentiel_distance_ville.csv")

DISTANCE_KEYS = [
    "VILLE_DEPART", "PAYS_DEPART", "VILLE_DESTINATION", "PAYS_DESTINATION"
]
DISTANCE_MEME_VILLE_KM = 8.0  # km estimés pour un déplacement dans la même ville

PROJECT_ROOT = os.path.abspath(os.path.join(BASE_DIR, "..", ".."))
CO2_TRANSPORTS_PATH = os.path.join(PROJECT_ROOT, "CO2_transports.tsv")
CO2_VOITURE_PATH = os.path.join(PROJECT_ROOT, "CO2_voiture.tsv")

COLS_MISSION_CALC = ["DISTANCE_KM", "CO2EQ", "IMPACT"]

MAP_TRANSPORT_CO2 = {
    "avion": "AVION",
    "plane": "AVION",
    "airplane": "AVION",
    "train": "TRAIN",
    "taxi": "TAXI",
    "transports en commun": "TC",
    "public transport": "TC",
}

# Colonnes à conserver par type de fichier
COLS_PERSONNEL = [
    'ID_PERSONNEL', 'NOM_PERSONNEL', 'PRENOM_PERSONNEL',
    'NUM_VOIE', 'CMPL_VOIE', 'CD_POSTAL', 'VILLE', 'PAYS',
    'FONCTION_PERSONNEL', 'TS_CREATION_PERSONNEL'
]

COLS_MATERIEL = [
    'ID_MATERIELINFO', 'ID_PERSONNEL', 'NOM_PERSONNEL', 'PRENOM_PERSONNEL',
    'DATE_ACHAT', 'TYPE', 'MODELE'
]

COLS_MISSION = [
    'ID_MISSION', 'ID_PERSONNEL', 'NOM_PERSONNEL', 'PRENOM_PERSONNEL',
    'DATE_MISSION', 'TYPE_MISSION', 'VILLE_DEPART', 'PAYS_DEPART',
    'VILLE_DESTINATION', 'PAYS_DESTINATION', 'TRANSPORT', 'ALLER_RETOUR'
]

# Dictionnaires de traductions (à compléter)
TRANSLATIONS_MATRIX = ['TYPE_MISSION', 'TRANSPORT']

LANGUE_CIBLE = "en"

In [24]:

# ==============================================================================
# SCHÉMA FLOCON — initialisé dans le main, réutilisé dans etl()
# ==============================================================================

# Initialisation des tables en tant que DataFrame PySpark Pandas

schema = {
    # Table de faits centrale
    'ALICIA_KEYS': ps.DataFrame(columns=[
        'ID_PERSONNEL', 'ID_MATERIELINFO', 'ID_MISSION'
    ]),
    # Dimensions
    'DF_PERSONNEL': ps.DataFrame(columns=COLS_PERSONNEL),
    'DF_MATERIEL':  ps.DataFrame(columns=COLS_MATERIEL + ['IMPACT']),
    'DF_MISSION':   ps.DataFrame(columns=COLS_MISSION + COLS_MISSION_CALC),
}

# ==============================================================================
# HELPERS
# ==============================================================================

def _filter_columns(df: ps.DataFrame, cols: list) -> ps.DataFrame:
    """Supprime toutes les colonnes qui ne sont pas dans 'cols'."""
    # En PySpark, on manipule directement la liste des colonnes pour le formatage
    df.columns = [str(c).strip() for c in df.columns]
    cols = [str(c).strip() for c in cols]
    
    existing = [c for c in cols if c in df.columns]
    missing  = [c for c in cols if c not in df.columns]
    if len(missing)>0:
        print(f"    [WARN] Colonnes attendues mais absentes : {missing}")
    return df[existing].copy()


def _append_unique(df_existing: ps.DataFrame, df_new: ps.DataFrame, pk: str) -> ps.DataFrame:
    """Ajoute df_new dans df_existing en dédupliquant sur la clé primaire 'pk'."""
    if df_new.empty:
        return df_existing
    combined = ps.concat([df_existing, df_new], ignore_index=True)
    return combined.drop_duplicates(subset=[pk], keep="last")


def _en_pandas(df):
    """pyspark.pandas → pandas (sinon copie pandas)."""
    if isinstance(df, pd.DataFrame):
        return df.copy()
    return df.to_pandas()


def _en_pyspark_pandas(df, etait_spark: bool):
    """pandas → pyspark.pandas si l'entrée venait de Spark."""
    if etait_spark:
        return ps.from_pandas(df)
    return df


def _charger_referentiel_impact():
    """Référentiel IMPACT : lookup par MODELE + repli « modèle par défaut » par TYPE."""
    ref = ps.read_csv(IMPACT_PATH, dtype=str)
    ref.columns = [str(c).strip().upper().replace("È", "E") for c in ref.columns]
    ref["MODELE"] = ref["MODELE"].str.strip()
    ref["TYPE"] = ref["TYPE"].str.strip()

    modele_defaut = (
        ref["MODELE"]
        .str.lower()
        .str.replace("è", "e", regex=False)
        .str.replace("é", "e", regex=False)
        == "modele par defaut"
    )
    impact_defaut_type = ref.loc[modele_defaut, ["TYPE", "IMPACT"]].drop_duplicates(
        subset=["TYPE"], keep="first"
    )
    impact_par_modele = ref.drop_duplicates(subset=["MODELE"], keep="first")[
        ["MODELE", "IMPACT"]
    ]
    return impact_par_modele, impact_defaut_type


def _joindre_impact_materiel(df_it, impact_par_modele, impact_defaut_type):
    """Jointure sur MODELE, puis IMPACT du « modèle par défaut » du TYPE si NaN."""
    etait_spark = hasattr(df_it, "to_pandas")
    df_it = _en_pandas(df_it)
    ref_modele = _en_pandas(impact_par_modele)
    ref_defaut = _en_pandas(impact_defaut_type)

    df_it["MODELE"] = df_it["MODELE"].str.strip()
    df_it["TYPE"] = df_it["TYPE"].str.strip()

    df_it = df_it.merge(ref_modele, on="MODELE", how="left")
    df_it = df_it.merge(
        ref_defaut.rename(columns={"IMPACT": "IMPACT_DEFAUT"}),
        on="TYPE",
        how="left",
    )
    df_it["IMPACT"] = df_it["IMPACT"].fillna(df_it["IMPACT_DEFAUT"])
    df_it = df_it.drop(columns=["IMPACT_DEFAUT"])
    return _en_pyspark_pandas(df_it, etait_spark)


def _compter_impact_vides(df: ps.DataFrame) -> int:
    if df.empty or "IMPACT" not in df.columns:
        return 0
    return int(df["IMPACT"].isna().sum())


# Cache géocodage (session notebook) pour limiter les appels Nominatim
_coords_cache: dict[tuple[str, str], tuple[float, float]] = {}
_geolocator = None


def _get_geolocator():
    global _geolocator
    if _geolocator is None:
        from geopy.geocoders import Nominatim
        _geolocator = Nominatim(user_agent="nf26_bges_etl")
    return _geolocator


def _charger_referentiel_distance() -> pd.DataFrame:
    cols = DISTANCE_KEYS + ["DISTANCE_KM"]
    if not os.path.exists(DISTANCE_REF_PATH):
        return pd.DataFrame(columns=cols)
    ref = pd.read_csv(DISTANCE_REF_PATH, sep=";")
    ref.columns = [str(c).strip() for c in ref.columns]
    for col in DISTANCE_KEYS:
        ref[col] = ref[col].astype(str).str.strip()
    ref["DISTANCE_KM"] = pd.to_numeric(ref["DISTANCE_KM"], errors="coerce")
    return ref


def _geocoder_ville(ville: str, pays: str):
    import time

    key = (str(ville).strip(), str(pays).strip())
    if key in _coords_cache:
        return _coords_cache[key]
    lieu = _get_geolocator().geocode(f"{key[0]}, {key[1]}")
    time.sleep(1)  # politique Nominatim
    if lieu is None:
        print(f"    [WARN] Géocodage impossible : {key[0]}, {key[1]}")
        return None
    _coords_cache[key] = (lieu.latitude, lieu.longitude)
    return _coords_cache[key]


def _calculer_distance_km(ville_dep, pays_dep, ville_dest, pays_dest):
    from geopy.distance import geodesic

    vd, pd_ = str(ville_dep).strip(), str(pays_dep).strip()
    va, pa = str(ville_dest).strip(), str(pays_dest).strip()
    if vd == va and pd_ == pa:
        return DISTANCE_MEME_VILLE_KM
    c1 = _geocoder_ville(vd, pd_)
    c2 = _geocoder_ville(va, pa)
    if c1 is None or c2 is None:
        return None
    return round(geodesic(c1, c2).km, 2)


def _ajouter_au_referentiel_distance(ref: pd.DataFrame, nouvelles_lignes: list[dict]) -> pd.DataFrame:
    if not nouvelles_lignes:
        return ref
    ref = pd.concat([ref, pd.DataFrame(nouvelles_lignes)], ignore_index=True)
    ref = ref.drop_duplicates(subset=DISTANCE_KEYS, keep="last")
    ref.to_csv(DISTANCE_REF_PATH, sep=";", index=False)
    return ref


def _joindre_distance_mission(
    df_mission: ps.DataFrame, ref_distance: pd.DataFrame
) -> tuple[ps.DataFrame, pd.DataFrame]:
    """Jointure sur le référentiel ; geopy + mise à jour du CSV si trajet inconnu."""
    if df_mission.empty:
        return df_mission, ref_distance

    etait_spark = hasattr(df_mission, "to_pandas")
    df = _en_pandas(df_mission)
    ref = ref_distance.copy()

    for col in DISTANCE_KEYS:
        df[col] = df[col].astype(str).str.strip()
        if not ref.empty:
            ref[col] = ref[col].astype(str).str.strip()

    df = df.merge(ref, on=DISTANCE_KEYS, how="left")

    manquant = df["DISTANCE_KM"].isna()
    if manquant.any():
        paires = df.loc[manquant, DISTANCE_KEYS].drop_duplicates()
        nouvelles = []
        for _, row in paires.iterrows():
            km = _calculer_distance_km(
                row["VILLE_DEPART"], row["PAYS_DEPART"],
                row["VILLE_DESTINATION"], row["PAYS_DESTINATION"],
            )
            if km is not None:
                nouvelles.append({
                    "VILLE_DEPART": row["VILLE_DEPART"],
                    "PAYS_DEPART": row["PAYS_DEPART"],
                    "VILLE_DESTINATION": row["VILLE_DESTINATION"],
                    "PAYS_DESTINATION": row["PAYS_DESTINATION"],
                    "DISTANCE_KM": km,
                })

        if nouvelles:
            ref_distance = _ajouter_au_referentiel_distance(ref_distance, nouvelles)
            print(f"    [OK] {len(nouvelles)} trajet(s) ajouté(s) à {DISTANCE_REF_PATH}")
            df = df.drop(columns=["DISTANCE_KM"])
            ref = ref_distance.copy()
            for col in DISTANCE_KEYS:
                ref[col] = ref[col].astype(str).str.strip()
            df = df.merge(ref, on=DISTANCE_KEYS, how="left")

    reste = int(df["DISTANCE_KM"].isna().sum())
    if reste:
        print(f"    [WARN] {reste} mission(s) sans DISTANCE_KM")

    return _en_pyspark_pandas(df, etait_spark), ref_distance


def standardize_timezone(df: ps.DataFrame, column: str) -> ps.DataFrame:
    """Convertit une colonne de dates en datetime standardisé."""
    if column in df.columns:
        # 1. On passe du monde "Pandas-on-Spark" au monde "Spark Natif"
        spark_df = df.to_spark()
        
        # 2. On utilise la fonction de conversion native de Spark (infaillible)
        spark_df = spark_df.withColumn(column, to_timestamp(col(column)))
        
        # 3. On revient dans le monde "Pandas-on-Spark"
        return spark_df.pandas_api()
        
    return df


def handle_missing_values(df, strategy="mean", target_col=None, feature_cols=None):
    """Complète les infos manquantes par moyenne ou régression linéaire"""
    if df.empty:
        return df

    if strategy == "mean" and target_col:
        df[target_col] = df[target_col].fillna(df[target_col].mean())

    elif strategy == "regression" and target_col and feature_cols:
        # Isolation des données incomplètes
        train_data = df.dropna(subset=feature_cols + [target_col])
        missing_data = df[df[target_col].isnull() & df[feature_cols].notnull().all(axis=1)]

        if not missing_data.empty and not train_data.empty:
            X_train = train_data[feature_cols]
            y_train = train_data[target_col]
            X_missing = missing_data[feature_cols]

            model = LinearRegression()
            model.fit(X_train, y_train)
            df.loc[missing_data.index, target_col] = model.predict(X_missing)

    return df
    


In [25]:
# Impact CO2 missions — sources : CO2_transports.tsv, CO2_voiture.tsv (kg eCO2/km)
"""
    Facteurs ADEME (kg eCO2 / km) source LABO1point5.
    Avion : 3 paliers selon DISTANCE_KM (<1000, 1000-3500, >3500).
    Train : Train <200 km si d<=200, sinon TGV >200 km.
    Taxi  : moyenne des voitures particulières (CO2_voiture.tsv).
    TC    : moyenne des bus (CO2_transports.tsv) — proxy transports en commun.
    """


_FACTEURS_CO2 = None


def _facteurs_co2() -> dict:
    """Charge une fois les facteurs ADEME (kg eCO2/km)."""
    global _FACTEURS_CO2
    if _FACTEURS_CO2 is not None:
        return _FACTEURS_CO2

    transports = pd.read_csv(CO2_TRANSPORTS_PATH, sep="\t")
    voitures = pd.read_csv(CO2_VOITURE_PATH, sep="\t")

    def facteur_transport(mot_cle):
        ligne = transports[transports["subsubcategory"].str.contains(mot_cle, na=False)]
        return float(ligne["total"].iloc[0])

    _FACTEURS_CO2 = {
        "avion_court": facteur_transport("Short haul"),      # < 1000 km
        "avion_moyen": facteur_transport("Medium haul"),     # 1000–3500 km
        "avion_long": facteur_transport("Long haul"),        # > 3500 km
        "train": facteur_transport("Train < 200"),           # <= 200 km
        "tgv": facteur_transport("TGV > 200"),               # > 200 km
        "taxi": float(voitures[voitures["subcategory"] == "Car"]["total"].mean()),
        "tc": float(transports[transports["subcategory"] == "Bus"]["total"].mean()),
    }
    return _FACTEURS_CO2


def _co2eq_km(transport: str, distance_km) -> float | None:
    """Facteur kg eCO2/km selon le transport et la distance."""
    if pd.isna(distance_km):
        return None

    code = MAP_TRANSPORT_CO2.get(str(transport).strip().lower())
    km = float(distance_km)
    f = _facteurs_co2()

    if code == "AVION":
        if km < 1000:
            return f["avion_court"]
        if km <= 3500:
            return f["avion_moyen"]
        return f["avion_long"]
    if code == "TRAIN":
        return f["train"] if km <= 200 else f["tgv"]
    if code == "TAXI":
        return f["taxi"]
    if code == "TC":
        return f["tc"]
    return None


def load_impact_mission(df_mission: ps.DataFrame) -> ps.DataFrame:
    """Ajoute CO2EQ (kg/km) et IMPACT (kg) = distance × CO2EQ × 2 si aller-retour."""
    if df_mission.empty:
        return df_mission

    etait_spark = hasattr(df_mission, "to_pandas")
    df = _en_pandas(df_mission)

    distance = pd.to_numeric(df["DISTANCE_KM"], errors="coerce")
    df["CO2EQ"] = [
        _co2eq_km(transport, km)
        for transport, km in zip(df["TRANSPORT"], distance)
    ]

    aller_retour = df["ALLER_RETOUR"].astype(str).str.lower().isin(["oui", "yes"])
    multiplicateur = np.where(aller_retour, 2.0, 1.0)

    df["CO2EQ"] = pd.to_numeric(df["CO2EQ"], errors="coerce").round(6)
    df["IMPACT"] = (distance * df["CO2EQ"] * multiplicateur).round(4)

    return _en_pyspark_pandas(df, etait_spark)

In [26]:
# À installer une fois si ce n'est pas fait : !pip install nest_asyncio

import nest_asyncio
nest_asyncio.apply()  # Cette ligne patch Jupyter pour autoriser asyncio.run()

In [ ]:

# ==============================================================================
# FONCTION ETL
# ==============================================================================


from googletrans import Translator

_translator = Translator()

def traduire_texte(texte, target=LANGUE_CIBLE):
    """Traduit une valeur scalaire (str) vers LANGUE_CIBLE (exécution driver uniquement)."""
    if texte is None or (isinstance(texte, float) and pd.isna(texte)):
        return texte
    if not isinstance(texte, str) or not texte.strip():
        return texte
    try:
        return _translator.translate(texte, dest=target).text
    except Exception as e:
        print(e)
        return texte

def traduire_dataframe(df: ps.DataFrame, cols: list[str]) -> ps.DataFrame:
    """Traduit les colonnes via un dictionnaire (compatible Spark, pas de .apply sur Series)."""
    for col_name in cols:
        if col_name not in df.columns:
            continue
        valeurs = df[col_name].dropna().unique().to_pandas()
        mapping = {
            v: traduire_texte(v)
            for v in valeurs
            if isinstance(v, str) and v.strip()
        }
        if mapping:
            df[col_name] = df[col_name].map(mapping)
    return df


def etl(current_date):
    date_str = current_date.strftime("%Y%m%d")
    print(f"--- Lancement ETL pour le jour : {date_str} ---")

    # Référentiels
    impact_par_modele, impact_defaut_type = _charger_referentiel_impact()
    print(f"[OK] Référentiel IMPACT chargé ({len(impact_par_modele)} modèles, {len(impact_defaut_type)} types par défaut)")
    ref_distance = _charger_referentiel_distance()
    print(f"[OK] Référentiel distance chargé ({len(ref_distance)} trajets)")
    sans_impact_jour = 0

    # Dictionnaires pour stocker les données du transformer
    missions_jour  = []
    info_jour      = []
    personnel_jour = []

    for site in SITES:

        # ── EXTRACTOR ─────────────────────────────────────────────────────────

        mission_path   = os.path.join(BASE_DIR, f"BDD_BGES_{site}", f"BDD_BGES_{site}_MISSION",
                                      f"MISSION_{date_str}.txt")
        it_path        = os.path.join(BASE_DIR, f"BDD_BGES_{site}", f"BDD_BGES_{site}_INFORMATIQUE",
                                      f"MATERIEL_INFORMATIQUE_{date_str}.txt")
        personnel_path = os.path.join(BASE_DIR, f"BDD_BGES_{site}",
                                      f"PERSONNEL_{site}.txt")

        print(f"\n  [SITE : {site}]")

        # ── MISSION ───────────────────────────────────────────────────────────
        if os.path.exists(mission_path):
            df_mission = ps.read_csv(mission_path, sep=';')

            # --- TRANSFORMER ---
            #traduire aller_retour
            df_mission = df_mission.drop_duplicates()
            df_mission = _filter_columns(df_mission, COLS_MISSION)
            df_mission = traduire_dataframe(df_mission, TRANSLATIONS_MATRIX)
            df_mission = standardize_timezone(df_mission, 'DATE_MISSION')
            df_mission, ref_distance = _joindre_distance_mission(df_mission, ref_distance)
            df_mission = load_impact_mission(df_mission)

            missions_jour.append(df_mission)
            print(f"    [OK] MISSION")
        else:
            print(f"    [SKIP] Pas de fichier MISSION pour ce jour")

        # ── MATÉRIEL INFORMATIQUE ─────────────────────────────────────────────
        if os.path.exists(it_path):
            df_it = ps.read_csv(it_path, sep=';')

            # --- TRANSFORMER ---

            #Traduire Type, modele
            df_it = df_it.drop_duplicates()
            df_it = _filter_columns(df_it, COLS_MATERIEL)
            df_it = standardize_timezone(df_it, 'DATE_ACHAT')
            df_it = _joindre_impact_materiel(df_it, impact_par_modele, impact_defaut_type)
            sans_impact_jour += _compter_impact_vides(df_it)

            info_jour.append(df_it)
            print(f"    [OK] MATERIEL INFORMATIQUE")
        else:
            print(f"    [SKIP] Pas de fichier MATERIEL pour ce jour")

        # ── PERSONNEL ─────────────────────────────────────────────────────────
        if os.path.exists(personnel_path):
            df_pers = ps.read_csv(personnel_path, sep=';')

            # --- TRANSFORMER ---
            #traduire fonction_personnel
            df_pers = df_pers.drop_duplicates()
            df_pers = _filter_columns(df_pers, COLS_PERSONNEL)
            df_pers = standardize_timezone(df_pers, 'TS_CREATION_PERSONNEL')

            personnel_jour.append(df_pers)
            print(f"    [OK] PERSONNEL")
        else:
            print(f"    [SKIP] Pas de fichier PERSONNEL pour ce site")

    # ── LOAD — Chargement dans le schéma flocon ────────────────────────────────

    if missions_jour or info_jour or personnel_jour:

        # Concaténation de tous les sites du jour (Opération optimisée dans Spark)
        df_all_missions  = ps.concat(missions_jour,  ignore_index=True) if missions_jour  else ps.DataFrame(columns=COLS_MISSION + COLS_MISSION_CALC)
        df_all_materiel  = ps.concat(info_jour,      ignore_index=True) if info_jour      else ps.DataFrame(columns=COLS_MATERIEL + ['IMPACT'])
        df_all_personnel = ps.concat(personnel_jour, ignore_index=True) if personnel_jour else ps.DataFrame(columns=COLS_PERSONNEL)

        df_all_missions.to_spark().show(5)
        df_all_materiel.to_spark().show(5)
        df_all_personnel.to_spark().show(5)

        if not df_all_missions.empty:
            schema["DF_MISSION"] = _append_unique(schema["DF_MISSION"], df_all_missions, pk="ID_MISSION")

        if not df_all_materiel.empty:
            schema["DF_MATERIEL"] = _append_unique(schema["DF_MATERIEL"], df_all_materiel, pk="ID_MATERIELINFO")

        if not df_all_personnel.empty:
            schema["DF_PERSONNEL"] = _append_unique(schema["DF_PERSONNEL"], df_all_personnel, pk="ID_PERSONNEL")

        # ==============================================================================
        # Table de faits ALICIA_KEYS — croisement des clés du jour
        # /!\ Les boucles for python imbriquées ont été remplacées par des "merge" (jointures) PySpark
        # ==============================================================================
        
        if not df_all_personnel.empty:
            # Extraction des clés uniques du personnel
            df_pers_keys = df_all_personnel[["ID_PERSONNEL"]].drop_duplicates()
            
            # Extraction des clés matériels existantes pour le jour
            if not df_all_materiel.empty:
                df_mat_keys = df_all_materiel[["ID_PERSONNEL", "ID_MATERIELINFO"]].drop_duplicates()
            else:
                df_mat_keys = ps.DataFrame(columns=["ID_PERSONNEL", "ID_MATERIELINFO"])
                
            # Extraction des clés missions existantes pour le jour
            if not df_all_missions.empty:
                df_mis_keys = df_all_missions[["ID_PERSONNEL", "ID_MISSION"]].drop_duplicates()
            else:
                df_mis_keys = ps.DataFrame(columns=["ID_PERSONNEL", "ID_MISSION"])
                
            # Jointure gauche 
            df_facts = df_pers_keys.merge(df_mat_keys, on="ID_PERSONNEL", how="left")
            df_facts = df_facts.merge(df_mis_keys, on="ID_PERSONNEL", how="left")

            if not df_facts.empty:
                combined_facts = ps.concat([schema["ALICIA_KEYS"], df_facts], ignore_index=True)
                schema["ALICIA_KEYS"] = combined_facts.drop_duplicates(
                    subset=["ID_PERSONNEL", "ID_MATERIELINFO", "ID_MISSION"], 
                    keep="last"
                )
                # print(f"[LOAD] ALICIA_KEYS   ← {len(df_facts)} ligne(s) ajoutée(s) ce jour")
                df_facts.to_spark().show(5)
    # Résumé
    print(f"\n{'─'*50}")
    print("Schéma flocon à jour (Note : les `.len()` forcent une évaluation sous PySpark)")
    print(f"{'─'*50}\n")
    return

# ==============================================================================
# MAIN
# ==============================================================================


def main():

    current_date = datetime(2026, 4, 29)
    end_date = datetime(2026, 11, 5)
    delta = timedelta(days=1)

    while current_date <= end_date:
      print(f"\nLancement du processus ETL pour le jour : {current_date.strftime('%Y-%m-%d')}")
      etl(current_date)
      current_date += delta

    print("Processus terminé avec succès.")


if __name__ == "__main__":

    main()


Lancement du processus ETL pour le jour : 2026-04-29
--- Lancement ETL pour le jour : 20260429 ---


/Users/alixaubert/Documents/NF26/Projet_BGES/.venv/lib/python3.11/site-packages/pyspark/pandas/strings.py:1608: FutureWarning: Default value of `regex` will be changed to `False` instead of `True` in 4.0.0.
  warnings.warn(


[OK] Référentiel IMPACT chargé (64 modèles, 16 types par défaut)
[OK] Référentiel distance chargé (192 trajets)

  [SITE : BERLIN]
    [OK] MISSION
    [OK] MATERIEL INFORMATIQUE
    [OK] PERSONNEL

  [SITE : LONDON]
    [OK] MISSION
    [OK] MATERIEL INFORMATIQUE
    [OK] PERSONNEL

  [SITE : LOSANGELES]
    [OK] MISSION
    [OK] MATERIEL INFORMATIQUE
    [OK] PERSONNEL

  [SITE : NEWYORK]
    [OK] MISSION
    [OK] MATERIEL INFORMATIQUE
    [OK] PERSONNEL

  [SITE : PARIS]
    [OK] MISSION
    [OK] MATERIEL INFORMATIQUE
    [OK] PERSONNEL

  [SITE : SHANGHAI]
    [OK] MISSION
    [OK] MATERIEL INFORMATIQUE
    [OK] PERSONNEL
+-----------------+--------------------+-------------+----------------+-------------------+----------------+------------+-----------+-----------------+----------------+---------+------------+-----------+------+---------+
|       ID_MISSION|        ID_PERSONNEL|NOM_PERSONNEL|PRENOM_PERSONNEL|       DATE_MISSION|    TYPE_MISSION|VILLE_DEPART|PAYS_DEPART|VILLE_DESTIN

+--------------------+---------------+----------+
|        ID_PERSONNEL|ID_MATERIELINFO|ID_MISSION|
+--------------------+---------------+----------+
|KeyPers_Berlin_12...|           NULL|      NULL|
|KeyPers_Berlin_12...|           NULL|      NULL|
|KeyPers_Berlin_12...|           NULL|      NULL|
|KeyPers_Berlin_12...|           NULL|      NULL|
|KeyPers_Berlin_12...|           NULL|      NULL|
+--------------------+---------------+----------+
only showing top 5 rows


──────────────────────────────────────────────────
Schéma flocon à jour (Note : les `.len()` forcent une évaluation sous PySpark)
──────────────────────────────────────────────────


Lancement du processus ETL pour le jour : 2026-04-30
--- Lancement ETL pour le jour : 20260430 ---


/Users/alixaubert/Documents/NF26/Projet_BGES/.venv/lib/python3.11/site-packages/pyspark/pandas/strings.py:1608: FutureWarning: Default value of `regex` will be changed to `False` instead of `True` in 4.0.0.
  warnings.warn(


...

+--------------------+--------------------+----------+
|        ID_PERSONNEL|     ID_MATERIELINFO|ID_MISSION|
+--------------------+--------------------+----------+
|KeyPers_Berlin_12...|                NULL|      NULL|
|KeyPers_Berlin_12...|                NULL|      NULL|
|KeyPers_Berlin_12...|                NULL|      NULL|
|KeyPers_Berlin_12...|                NULL|      NULL|
|KeyPers_Berlin_12...|BERLIN_MATERIEL_I...|      NULL|
+--------------------+--------------------+----------+
only showing top 5 rows


──────────────────────────────────────────────────
Schéma flocon à jour (Note : les `.len()` forcent une évaluation sous PySpark)
──────────────────────────────────────────────────


Lancement du processus ETL pour le jour : 2026-05-01
--- Lancement ETL pour le jour : 20260501 ---
[OK] Référentiel IMPACT chargé (64 modèles, 16 types par défaut)
[OK] Référentiel distance chargé (192 trajets)

  [SITE : BERLIN]
    [OK] MISSION
    [OK] MATERIEL INFORMATIQUE
    [OK] PERSONNEL

+--------------------+---------------+----------+
|        ID_PERSONNEL|ID_MATERIELINFO|ID_MISSION|
+--------------------+---------------+----------+
|KeyPers_Berlin_12...|           NULL|      NULL|
|KeyPers_Berlin_12...|           NULL|      NULL|
|KeyPers_Berlin_12...|           NULL|      NULL|
|KeyPers_Berlin_12...|           NULL|      NULL|
|KeyPers_Berlin_12...|           NULL|      NULL|
+--------------------+---------------+----------+
only showing top 5 rows


──────────────────────────────────────────────────
Schéma flocon à jour (Note : les `.len()` forcent une évaluation sous PySpark)
──────────────────────────────────────────────────


Lancement du processus ETL pour le jour : 2026-05-02
--- Lancement ETL pour le jour : 20260502 ---


/Users/alixaubert/Documents/NF26/Projet_BGES/.venv/lib/python3.11/site-packages/pyspark/pandas/strings.py:1608: FutureWarning: Default value of `regex` will be changed to `False` instead of `True` in 4.0.0.
  warnings.warn(


[OK] Référentiel IMPACT chargé (64 modèles, 16 types par défaut)
[OK] Référentiel distance chargé (192 trajets)

  [SITE : BERLIN]
    [OK] MISSION
    [OK] MATERIEL INFORMATIQUE
    [OK] PERSONNEL

  [SITE : LONDON]
    [OK] MISSION
    [OK] MATERIEL INFORMATIQUE
    [OK] PERSONNEL

  [SITE : LOSANGELES]


KeyboardInterrupt: 

In [ ]:
# ==============================================================================
# AUDIT — valeurs manquantes (par fichier + parcours BDD_BGES)
# ==============================================================================

import glob


def _serie_est_vide(serie: pd.Series) -> pd.Series:
    """NaN ou chaîne vide après strip (suffisant pour les fichiers BDD_BGES)."""
    return serie.isna() | (serie.astype(str).str.strip() == "")


def compter_manquantes_par_colonne(chemin_fichier: str, sep: str = ";") -> pd.DataFrame:
    """
    Pour un fichier .txt / .csv : compte les valeurs manquantes par colonne.

    Retourne un DataFrame (une ligne par colonne) :
      colonne, nb_manquantes, nb_lignes, pct_manquant
    """
    if not os.path.exists(chemin_fichier):
        raise FileNotFoundError(chemin_fichier)

    df = pd.read_csv(chemin_fichier, sep=sep, dtype=str)
    df.columns = [str(c).strip() for c in df.columns]
    n = len(df)
    if n == 0:
        return pd.DataFrame(columns=["colonne", "nb_manquantes", "nb_lignes", "pct_manquant"])

    lignes = []
    for nom_col in df.columns:
        nb = int(_serie_est_vide(df[nom_col]).sum())
        lignes.append({
            "colonne": nom_col,
            "nb_manquantes": nb,
            "nb_lignes": n,
            "pct_manquant": float(f"{100 * nb / n:.2f}") if n else 0.0,
        })
    return pd.DataFrame(lignes)


def lister_fichiers_bdd_bges(base_dir: str | None = None) -> list[str]:
    """Liste tous les .txt et .csv sous BDD_BGES (récursif)."""
    base = base_dir or BASE_DIR
    fichiers = []
    for pattern in ("**/*.txt", "**/*.csv"):
        fichiers.extend(glob.glob(os.path.join(base, pattern), recursive=True))
    return sorted(set(fichiers))


def _type_fichier(chemin: str) -> str:
    nom = os.path.basename(chemin).upper()
    if "MISSION_" in nom:
        return "MISSION"
    if "MATERIEL_INFORMATIQUE_" in nom:
        return "MATERIEL"
    if nom.startswith("PERSONNEL_"):
        return "PERSONNEL"
    if "IMPACT" in nom:
        return "IMPACT"
    return "AUTRE"


def auditer_bdd_bges(base_dir: str | None = None, verbose: bool = True) -> dict:
    """
    Parcourt tous les fichiers BDD_BGES et agrège les manquantes.

    Retourne :
      - detail : une ligne par (fichier, colonne) où nb_manquantes > 0
      - resume_colonnes : total de manquantes par nom de colonne (tous fichiers)
      - resume_types : total par type de fichier (MISSION, MATERIEL, …)
    """
    fichiers = lister_fichiers_bdd_bges(base_dir)
    detail_rows = []

    for chemin in fichiers:
        stats = compter_manquantes_par_colonne(chemin)
        avec_manquantes = stats[stats["nb_manquantes"] > 0]
        for _, row in avec_manquantes.iterrows():
            detail_rows.append({
                "fichier": chemin,
                "type_fichier": _type_fichier(chemin),
                "colonne": row["colonne"],
                "nb_manquantes": row["nb_manquantes"],
                "nb_lignes": row["nb_lignes"],
                "pct_manquant": row["pct_manquant"],
            })

    detail = pd.DataFrame(detail_rows)
    rapport = {
        "detail": detail,
        "fichiers_audites": len(fichiers),
    }

    if not detail.empty:
        rapport["resume_colonnes"] = (
            detail.groupby("colonne", as_index=False)
            .agg(nb_manquantes_total=("nb_manquantes", "sum"),
                 nb_fichiers_concernes=("fichier", "nunique"))
            .sort_values("nb_manquantes_total", ascending=False)
        )
        rapport["resume_types"] = (
            detail.groupby("type_fichier", as_index=False)
            .agg(nb_manquantes_total=("nb_manquantes", "sum"),
                 nb_fichiers_concernes=("fichier", "nunique"))
            .sort_values("nb_manquantes_total", ascending=False)
        )

    if verbose:
        print(f"Fichiers audités : {len(fichiers)}")
        print(f"Colonnes avec manquantes (fichier × colonne) : {len(detail)}")
        if not detail.empty:
            print("\n── Résumé par colonne (pour remplissage / suppression) ──")
            display(rapport["resume_colonnes"].head(30))
            print("\n── Résumé par type de fichier ──")
            display(rapport["resume_types"])
            print("\n── Détail (aperçu) ──")
            display(detail.head(30))

    return rapport


# ── Exemple : un seul fichier ──
# compter_manquantes_par_colonne(
#     os.path.join(BASE_DIR, "BDD_BGES_BERLIN", "PERSONNEL_BERLIN.txt")
# )

def comparer_jointure_impact():
    """Compare IMPACT vides : TYPE+MODELE vs MODELE seul vs MODELE + défaut par TYPE."""
    import glob

    impact_par_modele, impact_defaut_type = _charger_referentiel_impact()
    imp = pd.read_csv(IMPACT_PATH, dtype=str)
    imp.columns = [str(c).strip().upper().replace("È", "E") for c in imp.columns]
    imp["MODELE"] = imp["MODELE"].str.strip()
    imp["TYPE"] = imp["TYPE"].str.strip()

    fichiers = glob.glob(
        os.path.join(BASE_DIR, "**", "MATERIEL_INFORMATIQUE_*.txt"), recursive=True
    )
    deux_cols = modele_seul = avec_defaut = n = 0
    for chemin in fichiers:
        df = pd.read_csv(chemin, sep=";", dtype=str)
        df.columns = [str(c).strip() for c in df.columns]
        n += len(df)
        deux_cols += int(df.merge(imp, on=["TYPE", "MODELE"], how="left")["IMPACT"].isna().sum())
        modele_seul += int(
            df.assign(MODELE=df["MODELE"].str.strip())
            .merge(impact_par_modele, on="MODELE", how="left")["IMPACT"].isna().sum()
        )
        avec_defaut += _compter_impact_vides(
            _joindre_impact_materiel(df, impact_par_modele, impact_defaut_type)
        )

    print(f"Fichiers matériel : {len(fichiers)} | Lignes : {n}")
    print(f"IMPACT vide — TYPE + MODELE      : {deux_cols} ({100 * deux_cols / n:.2f} %)")
    print(f"IMPACT vide — MODELE seul        : {modele_seul} ({100 * modele_seul / n:.2f} %)")
    print(f"IMPACT vide — + défaut par TYPE  : {avec_defaut} ({100 * avec_defaut / n:.2f} %)")
    return {"type_et_modele": deux_cols, "modele_seul": modele_seul, "avec_defaut": avec_defaut, "lignes": n}


# ── Parcours complet BDD_BGES ──
# rapport_manquants = auditer_bdd_bges()

# ── Comparer les deux jointures IMPACT ──
test_impact = comparer_jointure_impact()


Fichiers audités : 2407
Colonnes avec manquantes (fichier × colonne) : 1413

── Résumé par colonne (pour remplissage / suppression) ──


,colonne,nb_manquantes_total,nb_fichiers_concernes
0,CMPL_VOIE,20572,6
1,IND_PAYS_NUM_TELP,20572,6
4,TYPE,1751,874
2,MODELE,705,521
3,NUM_SECU,225,6



── Résumé par type de fichier ──


,type_fichier,nb_manquantes_total,nb_fichiers_concernes
1,PERSONNEL,41369,6
0,MATERIEL,2456,1002



── Détail (aperçu) ──


,fichier,type_fichier,colonne,nb_manquantes,nb_lignes,pct_manquant
0,BDD_BGES/BDD_BGES/BDD_BGES_BERLIN/BDD_BGES_BER...,MATERIEL,TYPE,1,6,16.67
1,BDD_BGES/BDD_BGES/BDD_BGES_BERLIN/BDD_BGES_BER...,MATERIEL,MODELE,2,6,33.33
2,BDD_BGES/BDD_BGES/BDD_BGES_BERLIN/BDD_BGES_BER...,MATERIEL,TYPE,1,4,25.00
3,BDD_BGES/BDD_BGES/BDD_BGES_BERLIN/BDD_BGES_BER...,MATERIEL,TYPE,2,4,50.00
4,BDD_BGES/BDD_BGES/BDD_BGES_BERLIN/BDD_BGES_BER...,MATERIEL,MODELE,1,4,25.00
5,BDD_BGES/BDD_BGES/BDD_BGES_BERLIN/BDD_BGES_BER...,MATERIEL,TYPE,1,6,16.67
6,BDD_BGES/BDD_BGES/BDD_BGES_BERLIN/BDD_BGES_BER...,MATERIEL,MODELE,1,6,16.67
7,BDD_BGES/BDD_BGES/BDD_BGES_BERLIN/BDD_BGES_BER...,MATERIEL,TYPE,2,3,66.67
8,BDD_BGES/BDD_BGES/BDD_BGES_BERLIN/BDD_BGES_BER...,MATERIEL,TYPE,2,4,50.00
9,BDD_BGES/BDD_BGES/BDD_BGES_BERLIN/BDD_BGES_BER...,MATERIEL,MODELE,1,5,20.00
